In [32]:
import pandas as pd
import numpy as np


In [33]:
%cd /content/Walmart_sales_forecasting
save_file = "data/processed/sales_data_preprocessed.csv"

df_sales = pd.read_csv(save_file, parse_dates=['Date'])
df_sales

/content/Walmart_sales_forecasting


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.00,0.00,0.00,0.00,0.00,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.00,0.00,0.00,0.00,0.00,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.00,0.00,0.00,0.00,0.00,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.00,0.00,0.00,0.00,0.00,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.00,0.00,0.00,0.00,0.00,211.350143,8.106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,False,B,118221,64.88,3.997,4556.61,20.64,1.50,1601.01,3288.25,192.013558,8.684
421566,45,98,2012-10-05,628.10,False,B,118221,64.89,3.985,5046.74,0.00,18.82,2253.43,2340.01,192.170412,8.667
421567,45,98,2012-10-12,1061.02,False,B,118221,54.47,4.000,1956.28,0.00,7.89,599.32,3990.54,192.327265,8.667
421568,45,98,2012-10-19,760.01,False,B,118221,56.47,3.969,2004.02,0.00,3.18,437.73,1537.49,192.330854,8.667


In [34]:
df_feature = df_sales.copy()

In [35]:
test_split = pd.Timestamp("2012-08-05")
print(f'Test split day: {test_split}')
df_feature['is_test'] = df_feature['Date'] >= test_split
print(f'Num of test: {df_feature['is_test'].sum()}')
print(f'Num of train: {(~df_feature['is_test']).sum()}')



Test split day: 2012-08-05 00:00:00
Num of test: 35563
Num of train: 386007


In [36]:
temp_bins = [-np.inf, 18, 25, 32, np.inf]
temp_label = ['Cold', 'Cool', 'Warm', 'Hot']
df_feature['temp_category'] = pd.cut(df_feature['Temperature'], bins=temp_bins, labels=temp_label)
df_feature

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,is_test,temp_category
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.00,0.00,0.00,0.00,0.00,211.096358,8.106,False,Hot
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.00,0.00,0.00,0.00,0.00,211.242170,8.106,False,Hot
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.00,0.00,0.00,0.00,0.00,211.289143,8.106,False,Hot
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.00,0.00,0.00,0.00,0.00,211.319643,8.106,False,Hot
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.00,0.00,0.00,0.00,0.00,211.350143,8.106,False,Hot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,False,B,118221,64.88,3.997,4556.61,20.64,1.50,1601.01,3288.25,192.013558,8.684,True,Hot
421566,45,98,2012-10-05,628.10,False,B,118221,64.89,3.985,5046.74,0.00,18.82,2253.43,2340.01,192.170412,8.667,True,Hot
421567,45,98,2012-10-12,1061.02,False,B,118221,54.47,4.000,1956.28,0.00,7.89,599.32,3990.54,192.327265,8.667,True,Hot
421568,45,98,2012-10-19,760.01,False,B,118221,56.47,3.969,2004.02,0.00,3.18,437.73,1537.49,192.330854,8.667,True,Hot


In [37]:
df_feature['total_markdown'] = df_feature[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].sum(axis=1)
df_feature['avg_markdown'] = df_feature[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].mean(axis=1)
df_feature['max_markdown'] = df_feature[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].max(axis=1)

In [38]:
#change to categorical val
df_feature = pd.get_dummies(df_feature, columns=['IsHoliday', 'temp_category'], drop_first=True)
df_feature['Dept'] = df_feature['Dept'].astype('category')
df_feature['Store'] = df_feature['Store'].astype('category')
df_feature

,Store,Dept,Date,Weekly_Sales,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,CPI,Unemployment,is_test,total_markdown,avg_markdown,max_markdown,IsHoliday_True,temp_category_Cool,temp_category_Warm,temp_category_Hot
0,1,1,2010-02-05,24924.50,A,151315,42.31,2.572,0.00,0.00,...,211.096358,8.106,False,0.00,0.000,0.00,False,False,False,True
1,1,1,2010-02-12,46039.49,A,151315,38.51,2.548,0.00,0.00,...,211.242170,8.106,False,0.00,0.000,0.00,True,False,False,True
2,1,1,2010-02-19,41595.55,A,151315,39.93,2.514,0.00,0.00,...,211.289143,8.106,False,0.00,0.000,0.00,False,False,False,True
3,1,1,2010-02-26,19403.54,A,151315,46.63,2.561,0.00,0.00,...,211.319643,8.106,False,0.00,0.000,0.00,False,False,False,True
4,1,1,2010-03-05,21827.90,A,151315,46.50,2.625,0.00,0.00,...,211.350143,8.106,False,0.00,0.000,0.00,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,B,118221,64.88,3.997,4556.61,20.64,...,192.013558,8.684,True,9468.01,1893.602,4556.61,False,False,False,True
421566,45,98,2012-10-05,628.10,B,118221,64.89,3.985,5046.74,0.00,...,192.170412,8.667,True,9659.00,1931.800,5046.74,False,False,False,True
421567,45,98,2012-10-12,1061.02,B,118221,54.47,4.000,1956.28,0.00,...,192.327265,8.667,True,6554.03,1310.806,3990.54,False,False,False,True
421568,45,98,2012-10-19,760.01,B,118221,56.47,3.969,2004.02,0.00,...,192.330854,8.667,True,3982.42,796.484,2004.02,False,False,False,True


In [39]:
# One-hot encoding Type
df_feature = pd.get_dummies(df_feature, columns=['Type'], prefix='Type', drop_first=True)

In [40]:
df_feature = df_feature.sort_values(['Date', 'Store', 'Dept'])
df_feature['store_dept'] = ' store_' + df_feature['Store'].astype(str) + '_' + 'dept_' + df_feature['Dept'].astype(str)
df_feature

,Store,Dept,Date,Weekly_Sales,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,total_markdown,avg_markdown,max_markdown,IsHoliday_True,temp_category_Cool,temp_category_Warm,temp_category_Hot,Type_B,Type_C,store_dept
0,1,1,2010-02-05,24924.50,151315,42.31,2.572,0.00,0.00,0.0,...,0.00,0.000,0.00,False,False,False,True,False,False,store_1_dept_1
143,1,2,2010-02-05,50605.27,151315,42.31,2.572,0.00,0.00,0.0,...,0.00,0.000,0.00,False,False,False,True,False,False,store_1_dept_2
286,1,3,2010-02-05,13740.12,151315,42.31,2.572,0.00,0.00,0.0,...,0.00,0.000,0.00,False,False,False,True,False,False,store_1_dept_3
429,1,4,2010-02-05,39954.04,151315,42.31,2.572,0.00,0.00,0.0,...,0.00,0.000,0.00,False,False,False,True,False,False,store_1_dept_4
572,1,5,2010-02-05,32229.38,151315,42.31,2.572,0.00,0.00,0.0,...,0.00,0.000,0.00,False,False,False,True,False,False,store_1_dept_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421012,45,93,2012-10-26,2487.80,118221,58.85,3.882,4018.91,58.08,100.0,...,5247.26,1049.452,4018.91,False,False,False,True,True,False,store_45_dept_93
421146,45,94,2012-10-26,5203.31,118221,58.85,3.882,4018.91,58.08,100.0,...,5247.26,1049.452,4018.91,False,False,False,True,True,False,store_45_dept_94
421289,45,95,2012-10-26,56017.47,118221,58.85,3.882,4018.91,58.08,100.0,...,5247.26,1049.452,4018.91,False,False,False,True,True,False,store_45_dept_95
421434,45,97,2012-10-26,6817.48,118221,58.85,3.882,4018.91,58.08,100.0,...,5247.26,1049.452,4018.91,False,False,False,True,True,False,store_45_dept_97


In [41]:
# add month and day_of_week so that model can learn the pattern of time
df_feature['month'] = df_feature['Date'].dt.month
df_feature['week_of_year'] = df_feature['Date'].dt.isocalendar().week

In [42]:
lags = [2, 4, 6, 52] #add 52 because some dept sales have a yearly patern
for i in lags:
    df_feature[f'lags_{i}'] = df_feature.groupby('store_dept')['Weekly_Sales'].transform(lambda x : x.shift(i))
df_feature


,Store,Dept,Date,Weekly_Sales,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,temp_category_Hot,Type_B,Type_C,store_dept,month,week_of_year,lags_2,lags_4,lags_6,lags_52
0,1,1,2010-02-05,24924.50,151315,42.31,2.572,0.00,0.00,0.0,...,True,False,False,store_1_dept_1,2,5,NaN,NaN,NaN,NaN
143,1,2,2010-02-05,50605.27,151315,42.31,2.572,0.00,0.00,0.0,...,True,False,False,store_1_dept_2,2,5,NaN,NaN,NaN,NaN
286,1,3,2010-02-05,13740.12,151315,42.31,2.572,0.00,0.00,0.0,...,True,False,False,store_1_dept_3,2,5,NaN,NaN,NaN,NaN
429,1,4,2010-02-05,39954.04,151315,42.31,2.572,0.00,0.00,0.0,...,True,False,False,store_1_dept_4,2,5,NaN,NaN,NaN,NaN
572,1,5,2010-02-05,32229.38,151315,42.31,2.572,0.00,0.00,0.0,...,True,False,False,store_1_dept_5,2,5,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421012,45,93,2012-10-26,2487.80,118221,58.85,3.882,4018.91,58.08,100.0,...,True,True,False,store_45_dept_93,10,43,2644.24,2763.02,2541.07,1600.80
421146,45,94,2012-10-26,5203.31,118221,58.85,3.882,4018.91,58.08,100.0,...,True,True,False,store_45_dept_94,10,43,4041.28,4734.83,4477.17,5052.12
421289,45,95,2012-10-26,56017.47,118221,58.85,3.882,4018.91,58.08,100.0,...,True,True,False,store_45_dept_95,10,43,49334.77,49380.11,53711.96,52619.53
421434,45,97,2012-10-26,6817.48,118221,58.85,3.882,4018.91,58.08,100.0,...,True,True,False,store_45_dept_97,10,43,6463.32,6269.73,7004.70,5645.89


In [43]:
for window in lags:
    df_feature[f'mean_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    df_feature[f'max_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).max())
    df_feature[f'min_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).min())
    df_feature[f'std_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).std())

#exponential moving weight
for alpha in [0.5, 0.75]:
    df_feature[f'emw_sales_{alpha}'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).ewm(alpha=alpha, adjust=False).mean())

df_feature

,Store,Dept,Date,Weekly_Sales,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,mean_sales_last_6_week,max_sales_last_6_week,min_sales_last_6_week,std_sales_last_6_week,mean_sales_last_52_week,max_sales_last_52_week,min_sales_last_52_week,std_sales_last_52_week,emw_sales_0.5,emw_sales_0.75
0,1,1,2010-02-05,24924.50,151315,42.31,2.572,0.00,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
143,1,2,2010-02-05,50605.27,151315,42.31,2.572,0.00,0.00,0.0,...,24924.500000,24924.50,24924.50,NaN,24924.500000,24924.50,24924.50,NaN,24924.500000,24924.500000
286,1,3,2010-02-05,13740.12,151315,42.31,2.572,0.00,0.00,0.0,...,37764.885000,50605.27,24924.50,18159.046613,37764.885000,50605.27,24924.50,18159.046613,37764.885000,44185.077500
429,1,4,2010-02-05,39954.04,151315,42.31,2.572,0.00,0.00,0.0,...,29756.630000,50605.27,13740.12,18901.638325,29756.630000,50605.27,13740.12,18901.638325,25752.502500,21351.359375
572,1,5,2010-02-05,32229.38,151315,42.31,2.572,0.00,0.00,0.0,...,32305.982500,50605.27,13740.12,16253.555927,32305.982500,50605.27,13740.12,16253.555927,32853.271250,35303.369844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421012,45,93,2012-10-26,2487.80,118221,58.85,3.882,4018.91,58.08,100.0,...,17814.415000,54608.75,717.82,20287.128190,9460.779808,54608.75,23.92,12025.307089,35354.118359,45312.322469
421146,45,94,2012-10-26,5203.31,118221,58.85,3.882,4018.91,58.08,100.0,...,18109.411667,54608.75,1689.10,19999.636394,9309.340577,54608.75,23.92,12063.252477,18920.959180,13193.930617
421289,45,95,2012-10-26,56017.47,118221,58.85,3.882,4018.91,58.08,100.0,...,18695.113333,54608.75,2487.80,19466.945450,9336.483462,54608.75,23.92,12052.177317,12062.134590,7200.965154
421434,45,97,2012-10-26,6817.48,118221,58.85,3.882,4018.91,58.08,100.0,...,26666.748333,56017.47,2487.80,23647.747335,9909.124423,56017.47,23.92,13492.432928,34039.802295,43813.343789


In [44]:
# store level feature
df_feature['sum_store_1_week'] = df_feature.groupby(['Store', 'Date'])['Weekly_Sales'].transform('sum')
df_feature['mean_store_1_week'] = df_feature.groupby(['Store', 'Date'])['Weekly_Sales'].transform('mean')

# dept level feature
df_feature['sum_dept_1_week'] = df_feature.groupby(['Dept', 'Date'])['Weekly_Sales'].transform('sum')
df_feature['mean_dept_1_week'] = df_feature.groupby(['Dept', 'Date'])['Weekly_Sales'].transform('mean')
df_feature

,Store,Dept,Date,Weekly_Sales,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,mean_sales_last_52_week,max_sales_last_52_week,min_sales_last_52_week,std_sales_last_52_week,emw_sales_0.5,emw_sales_0.75,sum_store_1_week,mean_store_1_week,sum_dept_1_week,mean_dept_1_week
0,1,1,2010-02-05,24924.50,151315,42.31,2.572,0.00,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1643690.90,22516.313699,881833.41,19596.298000
143,1,2,2010-02-05,50605.27,151315,42.31,2.572,0.00,0.00,0.0,...,24924.500000,24924.50,24924.50,NaN,24924.500000,24924.500000,1643690.90,22516.313699,1997831.89,44396.264222
286,1,3,2010-02-05,13740.12,151315,42.31,2.572,0.00,0.00,0.0,...,37764.885000,50605.27,24924.50,18159.046613,37764.885000,44185.077500,1643690.90,22516.313699,484368.90,10763.753333
429,1,4,2010-02-05,39954.04,151315,42.31,2.572,0.00,0.00,0.0,...,29756.630000,50605.27,13740.12,18901.638325,25752.502500,21351.359375,1643690.90,22516.313699,1205801.77,26795.594889
572,1,5,2010-02-05,32229.38,151315,42.31,2.572,0.00,0.00,0.0,...,32305.982500,50605.27,13740.12,16253.555927,32853.271250,35303.369844,1643690.90,22516.313699,1116952.54,24821.167556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421012,45,93,2012-10-26,2487.80,118221,58.85,3.882,4018.91,58.08,100.0,...,9460.779808,54608.75,23.92,12025.307089,35354.118359,45312.322469,760281.43,11347.484030,1049693.34,24992.698571
421146,45,94,2012-10-26,5203.31,118221,58.85,3.882,4018.91,58.08,100.0,...,9309.340577,54608.75,23.92,12063.252477,18920.959180,13193.930617,760281.43,11347.484030,1291378.41,29349.509318
421289,45,95,2012-10-26,56017.47,118221,58.85,3.882,4018.91,58.08,100.0,...,9336.483462,54608.75,23.92,12052.177317,12062.134590,7200.965154,760281.43,11347.484030,3002617.27,66724.828222
421434,45,97,2012-10-26,6817.48,118221,58.85,3.882,4018.91,58.08,100.0,...,9909.124423,56017.47,23.92,13492.432928,34039.802295,43813.343789,760281.43,11347.484030,596563.70,13558.265909


In [45]:
print(df_feature.isna().sum())

Store                           0
Dept                            0
Date                            0
Weekly_Sales                    0
Size                            0
Temperature                     0
Fuel_Price                      0
MarkDown1                       0
MarkDown2                       0
MarkDown3                       0
MarkDown4                       0
MarkDown5                       0
CPI                             0
Unemployment                    0
is_test                         0
total_markdown                  0
avg_markdown                    0
max_markdown                    0
IsHoliday_True                  0
temp_category_Cool              0
temp_category_Warm              0
temp_category_Hot               0
Type_B                          0
Type_C                          0
store_dept                      0
month                           0
week_of_year                    0
lags_2                       6625
lags_4                      13134
lags_6        

In [46]:
nan_count = df_feature.isna().sum()
nan_columns = nan_count[nan_count > 0].index
nan_columns

Index(['lags_2', 'lags_4', 'lags_6', 'lags_52', 'mean_sales_last_2_week',
       'max_sales_last_2_week', 'min_sales_last_2_week',
       'std_sales_last_2_week', 'mean_sales_last_4_week',
       'max_sales_last_4_week', 'min_sales_last_4_week',
       'std_sales_last_4_week', 'mean_sales_last_6_week',
       'max_sales_last_6_week', 'min_sales_last_6_week',
       'std_sales_last_6_week', 'mean_sales_last_52_week',
       'max_sales_last_52_week', 'min_sales_last_52_week',
       'std_sales_last_52_week', 'emw_sales_0.5', 'emw_sales_0.75'],
      dtype='object')

In [47]:
nan_sample = df_feature.isna().sum(axis=1)
nan_sample = nan_sample[nan_sample > 0].count()
print(f'number of sample containing nan val: {nan_sample}')
print(f'number of total sample: {df_feature['Store'].count()}')
print(f'percent of nan sample: {nan_sample / df_feature['Store'].count()}')

number of sample containing nan val: 160487
number of total sample: 421570
percent of nan sample: 0.38068885357117443


# Decision: Drop Rows with Missing Values

In [48]:

df_feature = df_feature.dropna()
print(f'number of sample after drop: {df_feature['Store'].count()}')

number of sample after drop: 261083


In [49]:
# save data feather format
# feather faster and lighter than csv, no need to parse date like csv
%cd /content/Walmart_sales_forecasting
feather_dir = 'data/processed/feature_engineering.feather'
df_feature.to_feather(feather_dir)
 

/content/Walmart_sales_forecasting


In [50]:
%cd /content/Walmart_sales_forecasting/data/processed
!ls

/content/Walmart_sales_forecasting/data/processed
feature_engineering.feather  sales_data_preprocessed.csv
